# Thư viện và Dữ liệu

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import ParameterGrid
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
# tải dữ liệu giao dịch đầy đủ
df_fact = pd.read_csv('../data_raw/fact_sales_full.csv', parse_dates=['order_date'])
df_fact.head()

In [ ]:
# đổi tên biến
df_fact = df_fact.rename(columns={'line_total': 'revenue'})
df_fact.head(1)

# Chuẩn bị cho GBM

In [ ]:
# xây dựng RFM features
reference_date = pd.Timestamp('2026-03-31')
recency  = (df_fact.groupby('customer_code')['order_date'].max()
            .apply(lambda x: (reference_date - x).days).rename('recency_days'))
frequency = df_fact.groupby('customer_code')['so_number'].nunique().rename('frequency')
monetary  = df_fact.groupby('customer_code')['revenue'].sum().rename('monetary')
df_rfm = pd.concat([recency, frequency, monetary], axis=1).reset_index()

# last3m
last3m = df_fact[df_fact['order_date'] >= '2026-01-01']
last3m_cnt = last3m.groupby('customer_code')['so_number'].nunique().rename('last3m_orders')
df_rfm = df_rfm.merge(last3m_cnt.reset_index(), on='customer_code', how='left')
df_rfm['last3m_orders'] = df_rfm['last3m_orders'].fillna(0)
df_rfm.head()

In [ ]:
# đặc trưng và nhãn
feature_cols = ['recency_days','frequency','monetary','last3m_orders']
X = df_rfm[feature_cols].fillna(0)
y = (df_rfm['recency_days'] <= 90).astype(int)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"X shape: {X_scaled.shape}")
print(f"Tỉ lệ mua: {y.mean():.2%}")

In [ ]:
# tách tập train / future (chưa có dữ liệu Q2 nên dùng toàn bộ để train)
X_train, y_train = X_scaled, y

In [ ]:
# scale đặc trưng (giữ nguyên để predict)
print(f"Train shape: {X_train.shape}")

# Mô hình GBM Classifier

In [ ]:
# lấy tham số tốt nhất
parameters = pd.read_csv('best_params_classifier.csv', index_col=0)
parameters

In [ ]:
# trích xuất giá trị tham số
n_estimators     = int(parameters.loc['n_estimators'][0])
max_depth        = int(parameters.loc['max_depth'][0])
learning_rate    = float(parameters.loc['learning_rate'][0])
subsample        = float(parameters.loc['subsample'][0])
colsample_bytree = float(parameters.loc['colsample_bytree'][0])

In [ ]:
# mô hình GBM
model = XGBClassifier(n_estimators=n_estimators,
                      max_depth=max_depth,
                      learning_rate=learning_rate,
                      subsample=subsample,
                      colsample_bytree=colsample_bytree,
                      use_label_encoder=False,
                      eval_metric='logloss',
                      random_state=42)
model.fit(X_train, y_train)

# Dự báo

In [ ]:
# dự báo xác suất mua hàng Q2/2026
predictions_proba = model.predict_proba(X_scaled)[:, 1]
predictions_gbm   = pd.DataFrame({
    'customer_code': df_rfm['customer_code'],
    'gbm_proba':     predictions_proba
})

In [ ]:
# trực quan hóa
predictions_gbm['gbm_proba'].hist(bins=20, figsize=(10, 4))
plt.title('Phân bố xác suất mua hàng Q2/2026 theo đại lý')
plt.xlabel('Xác suất')
plt.tight_layout()
plt.show();

In [ ]:
# xuất kết quả
predictions_gbm.to_csv('Ensemble/predictions_gbm.csv', index=False)
print('✅ Đã lưu Ensemble/predictions_gbm.csv')